In [4]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# ============================================================
# DIRECTORY CONFIGURATION
# ============================================================

ROOT = Path.cwd()

SURROGATE_DIR = ROOT / "Data" / "runtime_data" / "surrogate_optimisation"
OUTPUT_DIR = ROOT / "Results" / "initial_final_fid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# INPUT FILES
# Expected structure:
#
# Data/runtime_data/surrogate_optimisation/
# ├── GHZ/
# │   ├── 4n_.json
# │   ├── 6n.json
# │   └── 8n.json
# ├── W/
# │   ├── 4n.json
# │   ├── 6n.json
# │   └── 8n.json
# └── LC/
#     ├── 4n.json
#     ├── 6n.json
#     └── 8n.json
# ============================================================

STATE_ORDER = ["GHZ", "W", "LC"]
NODE_ORDER = [4, 6, 8]

FILES_SURROGATE = {
    state: {
        n: SURROGATE_DIR / state / f"{n}n.json"
        for n in NODE_ORDER
    }
    for state in STATE_ORDER
}




In [1]:

print("Current directory:", ROOT)
print("Surrogate data directory:",SURROGATE_DIR)
print("Output directory:", OUTPUT_DIR)



# ============================================================
# USER SETTINGS
# ============================================================

VERSION = 2

SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SAVE_DPI = 700

USE_PYTHEUS_INITIAL = True
USE_PYTHEUS_FINAL = True


# ============================================================
# HISTOGRAM SETTINGS
# ============================================================

HIST_BINS = np.linspace(0.0, 1.0, 31)

HIST_RWIDTH = 1.0
HIST_EDGE_COLOR = "black"
HIST_LINEWIDTH = 1.4
HIST_ALPHA = 0.75

INITIAL_LABEL = "Initial graphs"
FINAL_LABEL = "Optimised graphs"

X_LIM = (-0.05, 1.05)

USE_LOG_Y = True
SHOW_GRID = False


# ============================================================
# FIGURE LAYOUT SETTINGS
# ============================================================

FIGSIZE = (18, 12)
INDIVIDUAL_FIGSIZE = (6.5, 5.0)

WSPACE = 0.32
HSPACE = 0.42

LEFT_MARGIN = 0.06
RIGHT_MARGIN = 0.99
BOTTOM_MARGIN = 0.07
TOP_MARGIN = 0.98


# ============================================================
# FONT AND STYLE SETTINGS
# ============================================================

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,

    "lines.linewidth": 2.0,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.unicode_minus": False,
})


# ============================================================
# EXPLICIT FONT-SIZE CONTROLS
# ============================================================

LABEL_FONTSIZE = 18
TICK_FONTSIZE = 15
PANEL_LEGEND_FONTSIZE = 13

CORNER_LABEL_FONTSIZE = 18

X_LABELPAD = 6
Y_LABELPAD = 6
TICK_PAD = 4


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_fidelities(summary_path: Path):
    """
    Load initial and final fidelities from optimisation summary JSON.
    """
    summary_path = Path(summary_path)

    if not summary_path.exists():
        raise FileNotFoundError(f"Missing file:\n{summary_path}")

    with open(summary_path, "r") as f:
        summary = json.load(f)

    if USE_PYTHEUS_INITIAL and "initial_pytheus_fidelities" in summary:
        initial_key = "initial_pytheus_fidelities"
    else:
        initial_key = "initial_fidelities"

    if USE_PYTHEUS_FINAL and "final_pytheus_fidelities" in summary:
        final_key = "final_pytheus_fidelities"
    elif "final_fidelities" in summary:
        final_key = "final_fidelities"
    elif "best_fidelities" in summary:
        final_key = "best_fidelities"
    elif "optimised_fidelities" in summary:
        final_key = "optimised_fidelities"
    elif "optimized_fidelities" in summary:
        final_key = "optimized_fidelities"
    else:
        raise KeyError(
            f"No final fidelity key found in {summary_path}.\n"
            "Expected one of: final_pytheus_fidelities, final_fidelities, "
            "best_fidelities, optimised_fidelities, optimized_fidelities."
        )

    if initial_key not in summary:
        raise KeyError(
            f"No initial fidelity key found in {summary_path}.\n"
            f"Expected key: {initial_key}"
        )

    initial = np.asarray(summary[initial_key], dtype=np.float64).ravel()
    final = np.asarray(summary[final_key], dtype=np.float64).ravel()

    valid = np.isfinite(initial) & np.isfinite(final)

    initial = initial[valid]
    final = final[valid]

    if len(initial) == 0:
        raise ValueError(f"No valid fidelity values found in {summary_path}")

    if initial.shape != final.shape:
        raise ValueError(
            f"Shape mismatch in {summary_path}: "
            f"{initial_key} shape = {initial.shape}, "
            f"{final_key} shape = {final.shape}"
        )

    return initial, final, initial_key, final_key, summary


def style_axis(ax):
    """
    Apply consistent paper-style axis formatting.
    """
    if SHOW_GRID:
        ax.grid(True, alpha=0.25, linewidth=0.6)
    else:
        ax.grid(False)

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONTSIZE,
        width=0.8,
        length=3.5,
        direction="out",
        pad=TICK_PAD,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.9)


def save_and_show_figure(fig, output_dir: Path, filename: str):
    """
    Save figure as PNG/PDF/SVG and show it in notebook.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = {}

    if SAVE_PNG:
        png_path = output_dir / f"{filename}.png"
        fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
        saved_paths["png"] = png_path.resolve()
        print(f"Saved PNG: {png_path.resolve()}")

    if SAVE_PDF:
        pdf_path = output_dir / f"{filename}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        saved_paths["pdf"] = pdf_path.resolve()
        print(f"Saved PDF: {pdf_path.resolve()}")

    if SAVE_SVG:
        svg_path = output_dir / f"{filename}.svg"
        fig.savefig(svg_path, bbox_inches="tight")
        saved_paths["svg"] = svg_path.resolve()
        print(f"Saved SVG: {svg_path.resolve()}")

    plt.show()
    plt.close(fig)

    return saved_paths


def collect_all_data():
    """
    Load all fidelity data into a nested dictionary.

    Returns:
        data[state][n] = {
            "initial": initial,
            "final": final,
            "initial_key": initial_key,
            "final_key": final_key,
            "summary_path": summary_path,
        }
    """
    data = {}
    all_stats = []

    for state in STATE_ORDER:
        data[state] = {}

        for n_nodes in NODE_ORDER:
            summary_path = FILES_SURROGATE[state][n_nodes]

            print("=" * 100)
            print(f"Loading: {state}, n={n_nodes}")
            print(f"File   : {summary_path}")

            if not summary_path.exists():
                print("WARNING: File does not exist. Skipping.")
                continue

            initial, final, initial_key, final_key, summary = load_fidelities(summary_path)

            data[state][n_nodes] = {
                "initial": initial,
                "final": final,
                "initial_key": initial_key,
                "final_key": final_key,
                "summary_path": str(summary_path),
            }

            stats = {
                "target": state,
                "n_nodes": int(n_nodes),
                "num_samples": int(len(initial)),
                "initial_key": initial_key,
                "final_key": final_key,
                "initial_mean": float(np.mean(initial)),
                "initial_std": float(np.std(initial)),
                "initial_min": float(np.min(initial)),
                "initial_max": float(np.max(initial)),
                "final_mean": float(np.mean(final)),
                "final_std": float(np.std(final)),
                "final_min": float(np.min(final)),
                "final_max": float(np.max(final)),
                "summary_path": str(summary_path),
            }

            all_stats.append(stats)

            print(f"Initial key  : {initial_key}")
            print(f"Final key    : {final_key}")
            print(f"Samples      : {len(initial)}")
            print(f"Initial mean : {np.mean(initial):.6f}")
            print(f"Final mean   : {np.mean(final):.6f}")

    return data, all_stats


# ============================================================
# MAIN 3x3 PLOTTING FUNCTION
# ============================================================
# ============================================================
# SHARED PANEL-DRAWING HELPER
# (used by both the combined 3x3 grid and the 9 individual figures,
# so both stay pixel-for-pixel consistent in bins/colours/alpha/
# log-scale/xlim/fonts/legend style)
# ============================================================

CORNER_BBOX = dict(
    boxstyle="round,pad=0.3",
    facecolor="white",
    edgecolor="none",
    alpha=1.0,
)


def plot_single_fidelity_panel(ax, state, n_nodes, initial, final):
    """Draw one initial-vs-optimised fidelity histogram panel on `ax`:
    histograms, log y-axis, per-panel x/y labels, a combined
    $\\mathrm{STATE}_{n}$ identifier near the upper-left corner, and a
    compact internal legend. Reused identically for the combined 3x3
    grid's cells and the 9 standalone individual figures.
    """
    ax.hist(
        initial,
        bins=HIST_BINS,
        alpha=HIST_ALPHA,
        edgecolor=HIST_EDGE_COLOR,
        linewidth=HIST_LINEWIDTH,
        rwidth=HIST_RWIDTH,
        label=INITIAL_LABEL,
    )

    ax.hist(
        final,
        bins=HIST_BINS,
        alpha=HIST_ALPHA,
        edgecolor=HIST_EDGE_COLOR,
        linewidth=HIST_LINEWIDTH,
        rwidth=HIST_RWIDTH,
        label=FINAL_LABEL,
    )

    if USE_LOG_Y:
        ax.set_yscale("log")

    ax.set_xlim(X_LIM)

    style_axis(ax)

    # Force tick labels -- sharex/sharey (used by the combined grid)
    # otherwise hides them on interior rows/columns, which would leave
    # the per-panel "Fidelity"/"Number of samples" labels floating above
    # a blank (unlabelled) axis. A no-op for the individual figures,
    # which don't share axes to begin with.
    ax.tick_params(labelbottom=True, labelleft=True)

    # Every panel gets its own axis labels -- no single shared
    # figure-level label, per an explicit request.
    ax.set_xlabel(
        "Fidelity",
        fontsize=LABEL_FONTSIZE,
        fontweight="bold",
        labelpad=X_LABELPAD,
    )
    ax.set_ylabel(
        "Number of samples",
        fontsize=LABEL_FONTSIZE,
        fontweight="bold",
        labelpad=Y_LABELPAD,
    )
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontweight("bold")

    # Combined state/n identifier, e.g. $\mathrm{GHZ}_{4}$, near the
    # upper-left corner. Opaque white box behind it for guaranteed
    # readability regardless of which bars happen to sit underneath.
    ax.text(
        0.04,
        0.94,
        rf"$\mathrm{{{state}}}_{{{n_nodes}}}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=CORNER_LABEL_FONTSIZE,
        fontweight="bold",
        bbox=CORNER_BBOX,
        zorder=10,
    )

    # Compact internal legend, in the gap between the "initial" bars
    # (fidelity 0.0-0.5) and the "optimised" spike (fidelity ~1.0) --
    # same position in every panel, combined or individual.
    ax.legend(
        loc="upper center",
        prop={"weight": "bold", "size": PANEL_LEGEND_FONTSIZE},
        frameon=True,
        framealpha=0.9,
        edgecolor="0.75",
        handlelength=1.2,
        borderpad=0.35,
        labelspacing=0.3,
        handletextpad=0.5,
    )


# ============================================================
# MAIN 3x3 PLOTTING FUNCTION
# ============================================================

def plot_3x3_initial_final_histograms(data):
    """
    Generate one combined 3x3 histogram figure.

    Rows:
        GHZ, W, LC

    Columns:
        n = 4, n = 6, n = 8
    """
    fig, axes = plt.subplots(
        len(STATE_ORDER),
        len(NODE_ORDER),
        figsize=FIGSIZE,
        sharex=True,
        sharey=True,
    )

    axes = np.asarray(axes)

    for row_idx, state in enumerate(STATE_ORDER):
        for col_idx, n_nodes in enumerate(NODE_ORDER):

            ax = axes[row_idx, col_idx]

            if state not in data or n_nodes not in data[state]:
                ax.axis("off")
                continue

            initial = data[state][n_nodes]["initial"]
            final = data[state][n_nodes]["final"]

            plot_single_fidelity_panel(ax, state, n_nodes, initial, final)

    fig.subplots_adjust(
        left=LEFT_MARGIN,
        right=RIGHT_MARGIN,
        bottom=BOTTOM_MARGIN,
        top=TOP_MARGIN,
        wspace=WSPACE,
        hspace=HSPACE,
    )

    saved_paths = save_and_show_figure(
        fig=fig,
        output_dir=OUTPUT_DIR,
        filename="initial_final_fidelity_combined",
    )

    return saved_paths


# ============================================================
# NINE INDIVIDUAL PLOTTING FUNCTION
# ============================================================

def plot_individual_fidelity_histograms(data):
    """
    Generate one standalone figure per (state, n) combination -- same
    bins/colours/alpha/log-scale/xlim/fonts/legend style as the combined
    grid's panels (both call plot_single_fidelity_panel), just sized and
    saved individually. No large external title -- the internal
    $\\mathrm{STATE}_{n}$ identifier is the only label.
    """
    all_saved_paths = {}

    for state in STATE_ORDER:
        for n_nodes in NODE_ORDER:

            if state not in data or n_nodes not in data[state]:
                print(f"WARNING: no data for {state}, n={n_nodes}. Skipping individual figure.")
                continue

            initial = data[state][n_nodes]["initial"]
            final = data[state][n_nodes]["final"]

            fig, ax = plt.subplots(figsize=INDIVIDUAL_FIGSIZE)

            plot_single_fidelity_panel(ax, state, n_nodes, initial, final)

            fig.tight_layout()

            filename = f"initial_final_fidelity_{state}_{n_nodes}"
            saved_paths = save_and_show_figure(
                fig=fig,
                output_dir=OUTPUT_DIR,
                filename=filename,
            )

            all_saved_paths[f"{state}_{n_nodes}"] = saved_paths

    return all_saved_paths


# ============================================================
# MAIN SCRIPT
# ============================================================

data, all_stats = collect_all_data()

combined_saved_paths = plot_3x3_initial_final_histograms(data)
individual_saved_paths = plot_individual_fidelity_histograms(data)

stats_path = OUTPUT_DIR / f"initial_final_histogram_statistics_3x3_v{VERSION}.json"

with open(stats_path, "w") as f:
    json.dump(all_stats, f, indent=4)

print("=" * 100)
print(f"Saved statistics: {stats_path.resolve()}")

if "png" in combined_saved_paths:
    print(f"Saved combined PNG: {combined_saved_paths['png']}")
if "pdf" in combined_saved_paths:
    print(f"Saved combined PDF: {combined_saved_paths['pdf']}")
if "svg" in combined_saved_paths:
    print(f"Saved combined SVG: {combined_saved_paths['svg']}")

print("-" * 100)
print("Saved individual figures:")
for key, paths in individual_saved_paths.items():
    if "png" in paths:
        print(f"  {key} PNG: {paths['png']}")
    if "pdf" in paths:
        print(f"  {key} PDF: {paths['pdf']}")
    if "svg" in paths:
        print(f"  {key} SVG: {paths['svg']}")

print("=" * 100)


Current directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code
Surrogate data directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/runtime_data/surrogate_optimisation
Output directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Results/initial_final_fid
Loading: GHZ, n=4
File   : /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/runtime_data/surrogate_optimisation/GHZ/4n.json
Initial key  : initial_fidelities
Final key    : final_pytheus_fidelities
Samples      : 1000
Initial mean : 0.167478
Final mean   : 0.999991
Loading: GHZ, n=6
File   : /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/runtime_data/surrogate_optimisation/GHZ/6n.json
Initial key  : initial_fidelities
Final key    : final_pytheus_fidelities
Samples      : 1000
Initial mean : 0.161499
Final mean   : 0.999991
Loading: GHZ, n=8
File   : /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/runt